In [1]:
print('hi')

hi


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
default_model = os.getenv('OPENAI_DEFAULT_MODEL', 'gpt-4o-mini')

# 실제 파이썬 함수 정의 (mock 날씨 데이터)
def get_weather(city: str) -> str:
    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "부산": "흐리고 기온은 21도입니다.",
        "제주": "비가 오고 기온은 18도입니다.",
    }
    return weather_data.get(city, f"{city}의 날씨 정보를 찾을 수 없습니다.")

# tool_map: 함수 이름 → 실제 함수 매핑
tool_map = {"get_weather": get_weather}

# 툴 정의
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 조회할 도시 이름"}
            },
            "required": ["city"]
        }
    }
]

city = "서울"
input_msg = [{'role': 'user', 'content': f'{city}의 오늘 날씨를 알려줘'}]

# 1) 1차 호출 : 모델이 어떤 툴을 쓸지 결정
resp1 = client.responses.create(
    model=default_model,
    input=input_msg,
    tools=tools,
    tool_choice='auto'
)

# 2) 함수 실행
tool_outputs = []
for item in resp1.output:
    if item.type == "function_call":
        func_name = item.name
        args = json.loads(item.arguments)

        # tool_map에서 실제 파이썬 함수 찾아 실행
        # **args는 딕셔너리 언패킹
        result = tool_map[func_name](**args)

        # 실행 결과를 function_call_output 형태로 저장
        # call_id는 어떤 function_call에 대한 결과인지 연결해주는 ID
        tool_outputs.append(
            {"type": "function_call_output",
             "call_id": item.call_id,
             'output': result}
        )

# 3) 2차 요청 : 함수 실행 결과를 모델에 다시 전달
if tool_outputs:                      # tool이 호출된 경우
    resp2 = client.responses.create(  # 2차 요청: 함수 실행 결과를 모델에 다시 전달
        model=default_model,
        input=input_msg + resp1.output + tool_outputs
        # 원래 입력 + 1차 모델 출력 + 함수 실행 결과를 함께 전달
        # 이렇게 해야 모델이 최종 자연어 답변을 생성할 수 있음
    )
    print(resp2.output_text)          # 최종 자연어 응답 출력
else:
    print(resp1.output_text)          # tool call 없이 바로 답변이 나온 케이스

서울의 오늘 날씨는 맑고 기온은 24도입니다. 좋은 하루 되세요!


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

# 1. Vector Store 생성
vector_store = client.vector_stores.create(
    name="PDF 저장소"
)

# 2. a.pdf 업로드 + Vector Store에 등록
file_batch = client.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vector_store.id,
    files=[open("Ai Agents Characteristics And Impact Essay.pdf", "rb")]
)

print("업로드 상태:", file_batch.status)
print("파일 개수:", file_batch.file_counts)

# 3. File Search로 PDF 검색/요약
response = client.responses.create(
    model="gpt-4o-mini",
    tools=[{
        "type": "file_search",
        "vector_store_ids": [vector_store.id]
    }],
    input="업로드한 PDF를 5줄로 요약해줘."
)

print(response.output_text)

In [ ]:
from openai import OpenAI
from google import genai
from dotenv import load_dotenv
import os

load_dotenv(override=True)

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
upstage_client = OpenAI(
    api_key=os.getenv('UPSTAGE_API_KEY'),
    base_url="https://api.upstage.ai/v1"
)

# 비교할 질문
question = "."

print("=" * 60)
print(f"질문: {question}")
print("=" * 60)

# OpenAI 응답
openai_resp = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}]
)
print("\n[GPT-4o-mini 답변]")
print(openai_resp.choices[0].message.content)

# Gemini 응답
gemini_resp = gemini_client.models.generate_content(
    model="gemini-2.0-flash",
    contents=question
)
print("\n[Gemini-2.0-flash 답변]")
print(gemini_resp.text)

# Upstage Solar 응답
upstage_resp = upstage_client.chat.completions.create(
    model="solar-pro",
    messages=[{"role": "user", "content": question}]
)
print("\n[Upstage Solar-Pro 답변]")
print(upstage_resp.choices[0].message.content)


In [ ]:
import yt_dlp
from openai import OpenAI
from dotenv import load_dotenv
import os
import glob

load_dotenv(override=True)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

url = "https://www.youtube.com/watch?v=ha9kn0qe4Mc"

# 1. 유튜브 오디오 다운로드 (ffmpeg 없이 m4a/webm 그대로)
print("1. 유튜브 오디오 다운로드 중...")
ydl_opts = {
    'format': 'bestaudio[ext=m4a]/bestaudio[ext=webm]/bestaudio',
    'outtmpl': 'temp_audio.%(ext)s',
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

# 다운로드된 파일 찾기
audio_files = glob.glob("temp_audio.*")
audio_file = audio_files[0] if audio_files else None
print(f"다운로드 완료: {audio_file}")

# 2. Whisper로 텍스트 변환
print("2. Whisper로 텍스트 변환 중...")
with open(audio_file, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        language="ko"
    )

print("\n=== 변환된 텍스트 ===")
print(transcript.text)
